# 11. Load Sequence Example: RNN vs LSTM vs GRU vs MonarchLinear

This notebook demonstrates how to:
1. Load a dataset as a sequence using `load_data_as_sequence`
2. Train and compare RNN, LSTM, and GRU models on the sequence data
3. Build an iterative neural network using `Sequential2D` with `MonarchLinear` blocks
4. Iterate the Sequential2D model over multiple time steps

We use MNIST as our dataset, treating each row of pixels (28 values) as one timestep in a sequence of length 28.

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from generatedata.load_data import load_data_as_sequence

from iterativennsimple.MonarchLinear import MonarchLinear
from iterativennsimple.Sequential1D import Sequential1D
from iterativennsimple.Sequential2D import Sequential2D, Identity

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

## 1. Load Data as a Sequence

We use `load_data_as_sequence` to load MNIST and reshape it into a sequence.

- **step_size=28**: Each timestep contains one row of the 28×28 image (28 pixel values).
- **label_every_step=True**: The ground-truth one-hot label (10 dimensions) is appended to each timestep's input. This gives the model access to the label at every step — useful for the iterative model to refine its predictions.

The result is:
- `X_seq`: shape `(N, 28, 38)` — 28 timesteps, each with 28 pixels + 10 label values
- `labels`: shape `(N, 10)` — one-hot encoded class labels

In [ ]:
# Load MNIST as a sequence
X_seq, labels = load_data_as_sequence(
    name="MNIST",
    step_size=28,
    label_every_step=True,
)

# Convert to PyTorch tensors
X_seq = torch.from_numpy(X_seq.astype(np.float32))
labels = torch.from_numpy(labels.astype(np.float32))

# Integer class labels for CrossEntropyLoss
y_cls = labels.argmax(dim=1)  # (N,)

# Dimensions
N = len(X_seq)
seq_len = X_seq.shape[1]
input_dim = X_seq.shape[2]     # step_size + label_dim = 28 + 10 = 38
label_dim = labels.shape[1]    # 10 for MNIST

print(f"Dataset: MNIST")
print(f"  N={N}, seq_len={seq_len}, input_dim={input_dim}, label_dim={label_dim}")
print(f"  X_seq shape: {X_seq.shape}")
print(f"  labels shape: {labels.shape}")

In [ ]:
# Train/validation split
val_fraction = 1 / 7
rng = np.random.default_rng(42)
idx = rng.permutation(N)
n_val = max(1, int(N * val_fraction))
val_idx, train_idx = idx[:n_val], idx[n_val:]

X_train, y_train = X_seq[train_idx], y_cls[train_idx]
X_val, y_val = X_seq[val_idx], y_cls[val_idx]

print(f"  train={len(y_train)}, val={len(y_val)}")

# DataLoaders
batch_size = 256
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=batch_size, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=batch_size)

## Hyperparameters

We use small models and few epochs so the notebook runs quickly. These are not tuned for best accuracy — the goal is to demonstrate the workflow.

In [ ]:
# Shared hyperparameters
hidden_size = 64   # Hidden state size for RNN/LSTM/GRU and MonarchLinear
num_epochs = 10
learning_rate = 1e-3

## 2. Baseline Models: RNN, LSTM, and GRU

We define three simple recurrent models. Each processes the sequence one timestep at a time and produces a classification from the hidden state at the **last** timestep.

All three models share the same structure:
1. A recurrent layer (`nn.RNN`, `nn.LSTM`, or `nn.GRU`) that processes the sequence
2. A linear layer that maps the final hidden state to class logits

In [ ]:
class SimpleRecurrentClassifier(nn.Module):
    """A simple sequence classifier using RNN, LSTM, or GRU.

    Args:
        rnn_type: One of "RNN", "LSTM", or "GRU".
        input_dim: Number of features per timestep.
        hidden_size: Hidden state dimension.
        label_dim: Number of output classes.
    """

    def __init__(self, rnn_type, input_dim, hidden_size, label_dim):
        super().__init__()
        # Select the recurrent layer type
        rnn_cls = {"RNN": nn.RNN, "LSTM": nn.LSTM, "GRU": nn.GRU}[rnn_type]
        self.rnn = rnn_cls(input_dim, hidden_size, num_layers=1, batch_first=True)
        self.fc = nn.Linear(hidden_size, label_dim)
        self.rnn_type = rnn_type

    def forward(self, x_seq):
        """
        Args:
            x_seq: (batch_size, seq_len, input_dim)
        Returns:
            logits: (batch_size, label_dim)
        """
        # Run the recurrent layer over the full sequence
        # Note: batch_first=True means input shape is (batch, seq_len, features)
        # For LSTM, _ captures (h_n, c_n); for RNN/GRU, _ captures h_n
        output, _ = self.rnn(x_seq)  # output: (batch, seq_len, hidden_size)
        # Take the hidden state at the last timestep
        last_hidden = output[:, -1, :]  # (batch, hidden_size)
        # Map to class logits
        logits = self.fc(last_hidden)   # (batch, label_dim)
        return logits

### Training Loop

We define a simple training function that works for any model that takes `x_seq` as input and returns `logits`. We use `CrossEntropyLoss` and the `Adam` optimizer.

In [ ]:
def train_model(model, train_loader, val_loader, num_epochs, learning_rate, device):
    """Train a model and print train/val accuracy each epoch.

    Args:
        model: Any nn.Module with forward(x_seq) -> logits.
        train_loader: DataLoader for training data.
        val_loader: DataLoader for validation data.
        num_epochs: Number of training epochs.
        learning_rate: Learning rate for Adam optimizer.
        device: torch device (cpu or cuda).

    Returns:
        dict with 'train_acc', 'val_acc' (lists of per-epoch accuracies).
    """
    model = model.to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    history = {"train_acc": [], "val_acc": []}

    for epoch in range(1, num_epochs + 1):
        # --- Training ---
        model.train()
        correct, total = 0, 0
        for x_batch, y_batch in train_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)

            logits = model(x_batch)
            loss = criterion(logits, y_batch)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            correct += (logits.detach().argmax(1) == y_batch).sum().item()
            total += len(y_batch)
        train_acc = correct / total * 100

        # --- Validation ---
        model.eval()
        correct, total = 0, 0
        with torch.no_grad():
            for x_batch, y_batch in val_loader:
                x_batch, y_batch = x_batch.to(device), y_batch.to(device)
                logits = model(x_batch)
                correct += (logits.argmax(1) == y_batch).sum().item()
                total += len(y_batch)
        val_acc = correct / total * 100

        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"  Epoch {epoch:2d}/{num_epochs}  train_acc={train_acc:.1f}%  val_acc={val_acc:.1f}%")

    return history

### Train RNN, LSTM, and GRU

We create one model of each type and train them with the same hyperparameters.

In [ ]:
results = {}

for rnn_type in ["RNN", "LSTM", "GRU"]:
    print(f"\n{'='*50}")
    print(f"Training {rnn_type}")
    n_params = sum(p.numel() for p in SimpleRecurrentClassifier(rnn_type, input_dim, hidden_size, label_dim).parameters())
    print(f"  Parameters: {n_params:,}")
    print(f"{'='*50}")

    model = SimpleRecurrentClassifier(rnn_type, input_dim, hidden_size, label_dim)
    history = train_model(model, train_loader, val_loader, num_epochs, learning_rate, device)
    results[rnn_type] = history

print("\n--- Summary ---")
for name, hist in results.items():
    print(f"  {name}: best val_acc = {max(hist['val_acc']):.1f}%")